In [1]:
import math

import numpy as np  
import scipy
import torch
from sentence_transformers import SentenceTransformer


d:\OCRproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Obtain Vector Embeddings

To calculate distance and similarity metrics, we first need to generate vector embeddings for some text documents. Let’s begin by defining a few example documents:


In [2]:
# Example documents
documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.'
]


In [3]:
documents

['Bugs introduced by the intern had to be squashed by the lead developer.',
 'Bugs found by the quality assurance engineer were difficult to debug.',
 'Bugs are common throughout the warm summer months, according to the entomologist.',
 'Bugs, in particular spiders, are extensively studied by arachnologists.']

In [4]:
# Load a pre-trained model
model = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L6-v2")

d:\OCRproject\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\test\.cache\huggingface\hub\models--sentence-transformers--paraphrase-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5003.45it/s]


In [5]:
# Generate embeddings
embeddings = model.encode(documents)

In [6]:
embeddings.shape

(4, 384)

In [7]:
embeddings

array([[-0.22804369, -0.24647681, -0.00319296, ...,  0.4552815 ,
         0.6341973 ,  0.5375049 ],
       [-0.357916  , -0.32083985,  0.15963238, ..., -0.07050638,
         0.92750233,  0.3437727 ],
       [ 0.20302932, -0.2689857 ,  0.16285147, ..., -0.19650991,
        -0.03379847,  0.5956153 ],
       [-0.04264278, -0.45721596, -0.09526484, ..., -0.580307  ,
         0.17248413,  0.09127852]], shape=(4, 384), dtype=float32)

## L2 (Euclidean) Distance


The L2 (Euclidean) distance between 2 vectors $a$ and $b$ can be calculated using <br>
$$ \text{L2}(a,b) = \ \sqrt{\sum_{i=1}^n (a_i - b_i)^2} $$

Let's try to implement the L2 distance manually before looking at off-the-shelf solutions available from third party libraries.


In [10]:
def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)

In [11]:
euclidean_distance_fn(embeddings[0], embeddings[1])

5.961788251769392

In [14]:
l2_dist_manual = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])

l2_dist_manual

array([[0.        , 5.96178825, 7.33939807, 7.15578169],
       [5.96178825, 0.        , 7.76861502, 7.393591  ],
       [7.33939807, 7.76861502, 0.        , 5.91992735],
       [7.15578169, 7.393591  , 5.91992735, 0.        ]])

In [15]:
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
l2_dist_scipy

array([[0.        , 5.96178891, 7.33939897, 7.15578093],
       [5.96178891, 0.        , 7.76861571, 7.39359049],
       [7.33939897, 7.76861571, 0.        , 5.91992726],
       [7.15578093, 7.39359049, 5.91992726, 0.        ]])

In [16]:
np.allclose(l2_dist_manual, l2_dist_scipy)

True

## Dot Product Similarity and Distance

The dot product between vectors $a$ and $b$ is calculated using  <br>
$$ a \cdot b = \sum_{i=1}^{n} a_i b_i $$

Let's define a custom function for calculating the dot product between two vectors.


In [17]:
def dot_product_fn(vector1, vector2):
    return sum(x * y for x, y in zip(vector1, vector2))

In [18]:
dot_product_fn(embeddings[0], embeddings[1])

np.float32(18.535393)

In [19]:
dot_product_manual = np.empty([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        dot_product_manual[i,j] = dot_product_fn(embeddings[i], embeddings[j])

dot_product_manual

array([[33.74440384, 18.53539276,  8.56981277,  7.8309269 ],
       [18.53539276, 38.86930847,  7.88996315,  8.66339397],
       [ 8.56981277,  7.88996315, 37.26200867, 17.66955948],
       [ 7.8309269 ,  8.66339397, 17.66955948, 33.12265778]])